In [1]:
from qiskit import *
from qiskit.quantum_info import Pauli, SparsePauliOp
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.opflow import PauliSumOp
from VQE_and_QAOA import VQE_and_QAOA

import networkx as nx

from Function_gi_params import *

/home/yahui/miniconda3/envs/py37/lib/python3.7/site-packages/ipykernel_launcher.py:5: DeprecationWarning: The ``qiskit.opflow`` module is deprecated as of qiskit-terra 0.24.0. It will be removed no earlier than 3 months after the release date. For code migration guidelines, visit https://qisk.it/opflow_migration.
  """


In [78]:
def Hqubo(N, coeff_matrix, No, lam):
    print('No: ', No, ',  lam: ', lam)
    H = 0
    Id = SparsePauliOp.from_sparse_list([("I", [0], 1),], num_qubits=N)
    for i in range(N):
        xi = SparsePauliOp.from_sparse_list([("I", [0], 0.5), ("Z", [i], -0.5)], num_qubits=N)
        H += (coeff_matrix[i, i] + lam*(1-2*No)) * xi 
    for i in range(N):
        for j in range(i+1, N):
            xij = SparsePauliOp.from_sparse_list([("I", [0], 0.25), ("Z", [i], -0.25), ("Z", [j], -0.25), ("ZZ", [i, j], 0.25)], num_qubits=N)
            H += (coeff_matrix[i, j]+2*lam) * xij 
    H += lam * Id * No**2
    return H
def Hqubo2(N, coeff_matrix, No, lam, mu):
    print('No: ', No, ',  lam: ', lam)
    H = 0
    Np = 0
    Id = SparsePauliOp.from_sparse_list([("I", [0], 1),], num_qubits=N)
    for i in range(N):
        xi = SparsePauliOp.from_sparse_list([("I", [0], 0.5), ("Z", [i], -0.5)], num_qubits=N)
        H += (coeff_matrix[i, i] -mu) * xi 
        Np += xi
    for i in range(N):
        for j in range(i+1, N):
            xij = SparsePauliOp.from_sparse_list([("I", [0], 0.25), ("Z", [i], -0.25), ("Z", [j], -0.25), ("ZZ", [i, j], 0.25)], num_qubits=N)
            H += (coeff_matrix[i, j]) * xij 
    P = (Np - No * Id) @ (Np - No * Id)
    return (H + lam * P).simplify()

def HP(N, coeff_matrix, No, lam):
    print('N: ', N)
    print('No: ', No, ',  lam: ', lam)
    Hp = 0
    Id = SparsePauliOp.from_sparse_list([("I", [0], 1),], num_qubits=N)
    for i in range(N):
        xi = SparsePauliOp.from_sparse_list([("I", [0], 0.5), ("Z", [i], -0.5)], num_qubits=N)
        Hp += lam*(1 - 2*No) * xi 
    for i in range(N):
        for j in range(i+1, N):
            print('i, j: ', (i, j))
            xij = SparsePauliOp.from_sparse_list([("I", [0], 0.25), ("Z", [i], -0.25), ("Z", [j], -0.25), ("ZZ", [i, j], 0.25)], num_qubits=N)
            Hp += 2*lam * xij 
    Hp += lam*Id * No**2
    return Hp

# 
N = 12
H = Hqubo(12, qubo_matrix, 6, 100)
#H = Hqubo2(12, qubo_matrix, 6, 100, 0)
eigen_list = H.to_matrix(sparse=True).diagonal()
# print('H: ', H)

exp_list = eigen_list
        
exp_min = min(exp_list)
exp_max = max(exp_list)

ground_id_list = []
for i in range(len(exp_list)):
    if abs(exp_list[i] - exp_min) < 1e-8:
        ground_id_list.append(i)
for id in ground_id_list:
    print('ground state: ', np.binary_repr(id, N))
print('expmin: ', exp_min)

No:  6 ,  lam:  100


ground state:  000111000111
expmin:  (-42.397659999996904+0j)


In [79]:
N = 12
# H = Hqubo(12, qubo_matrix, 6, 100)
H = Hqubo2(12, qubo_matrix, 6, 100, 0)
eigen_list = H.to_matrix(sparse=True).diagonal()
# print('H: ', H)

exp_list = eigen_list
        
exp_min = min(exp_list)
exp_max = max(exp_list)

ground_id_list = []
for i in range(len(exp_list)):
    if abs(exp_list[i] - exp_min) < 1e-8:
        ground_id_list.append(i)
for id in ground_id_list:
    print('ground state: ', np.binary_repr(id, N))
print('expmin: ', exp_min)

No:  6 ,  lam:  100


ground state:  000111000111
expmin:  (-42.397659999999036+0j)


In [85]:
N = 12
No = 6
lam = 10
mu = 0
alpha = 0.1  #1, 0.01
shots = 1000
ansatz_type = 'qaoa'  ##'linear_cnot', 'parallel_cz'
tau = 0  ## tau only for warm start of structure-inspired ansatz
initialization = 'zero' ## 'random' or 'zero'
failnum = 0 # count number of run failed
for r in range(10):
    gi_shots = 1000  ## shots for eatimating the expectation values of local pauli operators in warm start
    if shots == 0:# exact simulation
        shots = None
        approximation = True
    else:#simulation with finite shots
        approximation = False
    print('~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~')
    print('\nN: {}, \nr: {}, \nalpha: {}, \nshots: {}, \nansatz: {}, \nlayer: {}, \ntau: {}, \ninitialization: {}'\
        .format(N, r, alpha, shots, ansatz_type, layer, tau, initialization))
    #endregion

    ### folder to save result
    data_dir = './data/data_shots_{}_layer_{}/alpha_{}/N_{}/r_{}/ansatz_type_{}/'\
        .format(shots, layer, alpha, N, r, ansatz_type)
    os.makedirs(data_dir, exist_ok=True)

    #region load qubo instances, get Hamiltonian and edge_coeff_dict

    qubo_matrix = np.loadtxt('./qubo/qubo_matrix.txt')
    # print(qubo_matrix[:N, :N])
    edge_coeff_dict = {}
    h_list = []
    J_list = []
    edge_list = []
    
    for i in range(N):
        edge_coeff_dict[(i, )] = qubo_matrix[i, i]
        h_list.append(qubo_matrix[i, i]+lam*(1-2*No))
    for i in range(N):
        for j in range(i+1, N):
            edge_coeff_dict[(i, j)] = qubo_matrix[i, j]
            J_list.append(qubo_matrix[i, j]+2*lam)
            edge_list.append((i, j))
    
    H = Hqubo(N, qubo_matrix, No, lam)

    eigen_list = H.to_matrix(sparse=True).diagonal()
    #endregion

    ## order for two-qubit gate in circuit
    pairs_all = list(itertools.chain.from_iterable(partition(N)))

    #region get initial parameters
    if (ansatz_type) == 'structure_like_qubo_YZ_2':
        if initialization == 'warm_start_measure':
            gi_file_path = data_dir + '{}_tau_{}.pkl'.format(initialization, tau)
            layers_edge_params_dict, params_init, layers_exp_poss_dict = get_good_initial_params_measure(\
                N, tau, layer, edge_coeff_dict, pairs_all, eigen_list, shots, approximation, gi_file_path)
            print('\nwarm start fidelity', list(layers_exp_poss_dict['l_'+str(layer)].items())[0])
        elif initialization == 'warm_start_analy':
            gi_file_path = data_dir + '{}_tau_{}.pkl'.format(initialization, tau)
            edge_params_dict, params_init, layers_exp_poss_dict = get_good_initial_params_analy(\
                N, tau, layer, edge_coeff_dict, pairs_all, eigen_list, gi_file_path)
            print('\nwarm start fidelity', list(layers_exp_poss_dict['l_'+str(layer)].items())[0])
        elif initialization == 'zeros':
            params_init = np.zeros((N + 2*len(edge_list)) * layer)
        elif initialization == 'random':
            params_init = np.random.uniform(-np.pi, np.pi, (N + 2*len(edge_list)) * layer)
        else:
            raise ValueError('initialization method not found')
    elif (ansatz_type) == 'qaoa':
        if initialization == 'zeros':
            params_init = np.zeros(2 * layer) 
        elif initialization == 'random':
            params_init = np.random.uniform(-np.pi, np.pi, 2 * layer)
        else:
            raise ValueError('initialization method not found')
    elif (ansatz_type) == 'ma_qaoa':
        nparas_layer = N + len(list(edge_coeff_dict.keys()))
        if initialization == 'zeros':
            params_init = np.zeros(nparas_layer * layer) 
        elif initialization == 'random':
            params_init = np.random.uniform(-np.pi, np.pi, nparas_layer * layer)
        else:
            raise ValueError('initialization method not found')
    else: # for efficient su2 ansatz, 
        #layer should be (1 + 2*len(edge_list)/N) times more than ansatz 'structure_like_qubo_YZ_2' to have the same number of parameters
        if initialization == 'zeros':
            params_init = np.zeros(N * layer) 
        elif initialization == 'random':
            params_init = np.random.uniform(-np.pi, np.pi, N * layer)
        else:
            raise ValueError('initialization method not found')
    #endregion
    print('\ninitial parameters: ', params_init)




    vqe = VQE_and_QAOA(Hamiltonian = H, n_qubits = N, ansatz_type = ansatz_type, alpha = alpha, circuit_show = False, shots = shots)
    vqe.edge_coeff_dict = edge_coeff_dict

    print('vqe.shots: ', vqe.shots)
    print('vqe.anzatz_type: ', vqe.ansatz_type)
    E_min, E_max, ground_id_list = vqe.Get_minimun_from_H_mat()
    print('E_min from ED: ', E_min)
    for id in ground_id_list:
        print('ground state: ', np.binary_repr(id, N))
        Nnum = 0
        for s in np.binary_repr(id, N):
            Nnum += int(s)
        print('number of 1 in bitstring: ', Nnum)
    

    ## set the gate order in circuit
    vqe.edge_list = pairs_all

    vqe.r_eval = []
    vqe.poss_eval= []
    vqe.cvar_eval = []
    vqe.std_eval = []
    final = minimize(vqe.CVaR_expectation,
                    params_init,
                    jac=False,
                    bounds=None,
                    method='COBYLA',
                    callback=None,
                    options={'maxiter': len(params_init)*5})

    save_list = np.array([vqe.cvar_eval, vqe.r_eval, vqe.poss_eval, vqe.std_eval])

    np.savetxt(data_dir + '/result_{}_tau_{}.txt'.format(initialization, tau), save_list.T)
    np.savetxt(data_dir + '/final_params_{}_tau_{}.txt'.format(initialization, tau), final.x)
    print('final fidelity: ', max(vqe.poss_eval))
    print('Write sucessfully to ' + data_dir)

    ### check the result
    print('vqe.poss_eval[0]', vqe.poss_eval[0])
    if max(vqe.poss_eval) < alpha:
        failnum += 1
        print('failnum: ', failnum)

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

N: 12, 
r: 0, 
alpha: 0.1, 
shots: None, 
ansatz: parallel_cz, 
layer: 3, 
tau: 0, 
initialization: random
No:  6 ,  lam:  10

initial parameters:  [-1.71651919 -2.38427414 -0.54101688  0.37270508 -2.41420184  2.94186552
 -0.05883746  2.89849556 -0.59723013  2.14053043 -0.16848224  0.7329524
  2.43946885 -3.11630861 -0.67487929 -1.02579223  2.78245026 -1.59870539
 -0.87191664 -0.62587448 -2.90374049  0.43360411 -2.76230566  2.05439054
 -1.58262699  0.29967138 -1.91498338  2.56122053  0.18764113  0.53499823
  2.45767639  2.06551943  1.44277023 -1.33012291  1.38027302  2.6779757 ]
vqe.shots:  None
vqe.anzatz_type:  parallel_cz
E_min from ED:  -42.3976599999998
ground state:  000111000111
number of 1 in bitstring:  6
final fidelity:  8.54813951364062e-05
Write sucessfully to ./data/data_shots_None_layer_3/alpha_0.1/N_12/r_0/ansatz_type_parallel_cz/
vqe.poss_eval[0] 1.0697833972740722e-05
failnum:  1
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

N: 12, 
r: 1, 
alpha: 0.1,

(I-Zi)/2 * (I-Zi)/2